# Demo notebook to create smaller, standardized and more usable .nc files.


In [3]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.append(os.path.abspath('../Pathfinder'))

from helper_functions import *
from radar_functions import *
from pathfinder import *
from attach_grounddata import *
from add_dataflashlog import *
from clicki_tool import *

%matplotlib inline

In [4]:
#TODO: translate internal layers into depth within snowpack [m]

### Load RADAR object

In [14]:
datasetID = '24_04_2025_4'
campaignID = '2025_St3TARTFO'

RADAR = load_RADAR(radar_type='UWiBaSS', datasetID=datasetID, campaignID=campaignID)

Dataset 24_04_2025_4 from campaign 2025_St3TARTFO is loaded from /Volumes/PortableSSD/SnowDrone/UWiBaSS/24_04_2025_4/uwibass_object.pkl


In [16]:
RADAR.target_type = 'sea_ice'

### Write everything to a .nc file

In [17]:
variables_dict = {
    'snow_depth': 'PF_snow_depth',
    'snow_depth_uncertainty': 'PF_total_uncertainty',
    'dielectric_constant': 'PF_snow_profile_eps_r',
    'dielectric_constant_uncertainty': 'PF_snow_profile_eps_r_uncertainty',
    'usage_mask': 'altitude_mask',
    'laser_altitude': 'CTUN_SAlt',
    'UTM_x': 'log_UTM_x',
    'UTM_y': 'log_UTM_y',
    'platform_yaw': 'yaw',
    'platform_roll': 'roll',
    'platform_pitch': 'pitch',
    'platform_pitch_compensated': 'compensated_pitch',
    
}

coordinates_dict = {
    'time': 'datetime_timestamp',
    'lat': 'GPS_Lat',
    'lon': 'GPS_Lng'
}

meta_dict = {
    'UTM_zone': RADAR.utm_zone,
    'datasetID': RADAR.dataset_name,
    'campaignID': RADAR.campaignID,
    'target_type': RADAR.target_type
}

In [18]:
ds = xr.Dataset(
    data_vars=dict(
      **{new_var: (["time"], getattr(RADAR, old_var)) for new_var, old_var in variables_dict.items()},
    ),
    coords=dict(
      time = ("time", getattr(RADAR, coordinates_dict["time"])),
      lat = ("time", getattr(RADAR, coordinates_dict["lat"])),
      lon = ("time", getattr(RADAR, coordinates_dict["lon"])),
    ),
    attrs=dict(
      **meta_dict
    )
)
ds.to_netcdf(os.path.join(RADAR.data_path, "Uwibass", datasetID, f"{datasetID}_outfile.nc"))